# Robot Simulation with PyCRAM: Object Manipulation and Movement
In this notebook, we will walk through a complete example of setting up a robot environment, creating objects, and moving the robot to detect and interact with those objects using **PyCRAM**. We will explain the code step by step and ensure you understand how each function works in the simulation context.



## Step 1: Initialization

We start by importing the necessary libraries to set up the robot simulation. Each library serves a specific role in controlling the robot, creating the world, and interacting with objects:
TFBroadcaster: This is responsible for managing transformations in the environment, enabling the robot to understand spatial relations.
VizMarkerPublisher: This is used to visualize markers that help track the robot and object locations within the simulated world.
BulletWorld: A class from the PyCRAM framework that helps to create and manage the simulation environment.
ActionDesignator, LocationDesignator, ObjectDesignator: These represent abstract designators for robot actions, locations, and objects respectively, helping to describe tasks and identify targets.
Pose: Represents the position and orientation of objects or robots in the simulated world.
SimulatedRobot: This enables running the robot in a simulated mode for testing without requiring a physical robot.

In [ ]:
from pycram.ros.tf_broadcaster import TFBroadcaster
from pycram.ros.viz_marker_publisher import VizMarkerPublisher, AxisMarkerPublisher
from pycram.worlds.bullet_world import BulletWorld
from pycram.designators.action_designator import *
from pycram.designators.location_designator import *
from pycram.designators.object_designator import *
from pycram.datastructures.enums import ObjectType, WorldMode, TorsoState
from pycram.datastructures.pose import Pose
from pycram.process_module import simulated_robot, with_simulated_robot
from pycram.object_descriptors.urdf import ObjectDescription
from pycram.world_concepts.world_object import Object
from pycram.datastructures.dataclasses import Color
from pycram.designators.object_designator import BelieveObject

extension = ObjectDescription.get_file_extension()
print("Ready for the next cell.")

## Step 2: Setting Up the Simulation World
In this step, we set up the simulation environment using `BulletWorld`. We are using the `DIRECT` world mode, which means the simulation runs without a GUI (graphical interface), making it faster and more suitable for headless operation. We also initialize a visualization marker publisher to track objects and visualize movements in the simulation.

- `BulletWorld(WorldMode.DIRECT)`: Creates a simulation world in headless mode (no visual interface).
- `VizMarkerPublisher()`: Visualizes the movements and positions of objects and robots in the simulation world.
TODO: rapid fire simulation robot is teleporting

In [ ]:
world = BulletWorld(WorldMode.DIRECT)
viz = VizMarkerPublisher()
tf = TFBroadcaster()
print("Ready for the next cell.")

## Step 3: Adding the Robot to the World
Now, we add the robot into the simulation. In this case, we use a PR2 robot model. We define the robot’s type and position in the world using a pose. The pose is a tuple of (x, y, z) coordinates that specify the robot's starting position.

- `Object(robot_name, ObjectType.ROBOT, ...)`: Defines the robot as an object in the world.
- `Pose([1, 2, 0])`: Sets the robot's initial position at coordinates (1, 2, 0).

In [ ]:
pr2 = Object("pr2", ObjectType.ROBOT, "pr2.urdf")     
apartment = Object('apartment', ObjectType.ENVIRONMENT, f'apartment-small{extension}')
milk = Object("milk", ObjectType.MILK, "milk.stl", pose=Pose([2.5, 2, 1.02]), color=Color(0, 0, 1, 1))
#You should see on the right side every object in the simulation, after that you can continue with the next cell

## Step 4: Understanding Action Designator (NavigateAction)
Action Designators are high-level descriptions of actions which the robot should execute.

Action Designators are created from an Action Designator Description, which describes the type of action as well as the parameter for this action. Parameter are given as a list of possible parameters. For example, if you want to describe the robot moving to a table you would need a NavigateAction and a list of poses that are near the table. The Action Designator Description will then pick one of the poses and return a performable Action Designator which contains the picked pose. A Navigation would look like this:
    
 ```python
 NavigateAction(target_locations=[Pose([1.5, 2, 0])]).resolve().perform()
 ```

To move the robot we need to create a description and resolve it to an actual Designator. The description of navigation only needs a list of possible poses.

In [ ]:
nav_pose = Pose([1.5, 2, 0])


with simulated_robot:
   NavigateAction(target_locations=[nav_pose]).resolve().perform()  
#If you see the robot moving you can continue with the next cell

Summary of what we did: 
  - create the pose where we want to move the robot
  - create a description describing a navigation with a list of possible poses (in this case the list contains only one pose) and
  - create an action designator from the description

The action designator contains the pose picked from the list of possible poses and can then be performed.


Every designator that is performed needs to be in an environment that specifies where to perform the designator: Either on the real robot or the simulated one. 
This environment is called simulated_robot. 
If we want to perform actions on the real robot we would use real_robot instead.

There are also decorators which do the same thing but for whole methods, they are called with_real_robot and with_simulated_robot.

## Step 5: Let the robot look at the object

The LookAtAction is used to make the robot look at a specific pose. Fill in the Pose at **#YOUR CODE** HERE and try to let the robot look at the milk on the counter top. Btw the robot neck joint is not continuous so it can't look at every point in space. 

In [ ]:
target_location = #YOUR CODE HERE
with simulated_robot:
    LookAtAction(targets=[target_location]).resolve().perform()

#If you see the robot looking at the milk you can continue with the next cell (he moved the head)

## Step 6: Detect the object



The robot is commanded to move to a target location, focus on a specific object (like a milk carton), and detect it. These tasks are managed using designators: `NavigateAction` for moving, `LookAtAction` for directing its gaze, and `DetectAction` for identifying objects. 

But how does the system actually identify the object? In the communication between PyCRAM and Robokudo, the semantic description of an object plays a crucial role. For instance, you can query the system with phrases like "detect an object that is blue" or "detect all objects that are round." To handle such open-ended descriptions, a `BeliefObject` designator is used. This designator allows the robot to describe and identify objects present in the `BulletWorld`—i.e., objects stored in its belief state, which is how it gets its name.


In [ ]:
undetermined_object_description = BelieveObject(names=["milk"])
print(undetermined_object_description.resolve())

The DetectAction is used to detect objects in the field of vision (FOV) of the robot, that fits the object description. The detect designator will return a resolved instance of an ObjectDesignatorDescription. So for the next step write the code to detect the object. This includes the following steps: 
- Park the robot arms
- Navigate to the object
- Look at the object
- Detect the object
- Print the detected object
- The object should be the milk carton on the counter top
Remember to use the decorator for the simulated robot environment. Tip for the Pose: You can use the pose of the milk object by resolving it and calling the pose on the instance.


In [ ]:
## add your code here

## Solution

<details>

<summary>Click here to get the solution</summary>

```python
with simulated_robot:
    ParkArmsAction([Arms.BOTH]).resolve().perform()

    NavigateAction([Pose([1.5, 2, 0], [0, 0, 0, 1])]).resolve().perform()

    LookAtAction(targets=[undetermined_object_description.resolve().pose]).resolve().perform()

    obj_desig = DetectAction(undetermined_object_description).resolve().perform()

    print(obj_desig)
```
</details>

## Step 8: Understanding the object designator
Object designators are used to describe objects located in the BulletWorld or the real environment and then resolve them during runtime to concrete objects.

Object designators are different from the Object class in bullet_world in the way that they just describe an object and do not create objects or provide methods to manipulate them. Nevertheless, object designators contain a reference to the BulletWorld object.

An Object designator takes two parameters, of which at least one has to be provided. These parameters are:

A list of names
A list of types
Object Designators work similar to Location designators, they get constrains describing a set of objects and when resolved return a specific instance.

One of the great features of PyCRAM is its flexibility in handling both simulated and real environments seamlessly. Regardless of whether we interact with objects in a simulation or the real world, the designator remains the same! In simulation, object detection doesn't rely on a real camera—instead, the system simulates the detection process. However, when using the same detection action on a real robot, PyCRAM integrates with tools like Robokudo to recognize objects in the physical environment. This capability allows us to easily switch between simulation and reality using the same code, which makes development highly efficient. That said, it's important to remember that the real world is much more complex than the simulation, which is only a simplified model. Nonetheless, the ability to use the same code for both simulated and real robots significantly simplifies transitioning between the two environments.

## Step 9: Now have fun in the next part!

For Hands-On Exercise 4, please use this Virtual Lab or go back to the main page: [Robokudo Lab Tutorials](https://binder.intel4coro.de/v2/git/https%3A%2F%2Fgitlab.informatik.uni-bremen.de%2Ffmuehlis%2Frobokudo-lab.git/tutorials)


